# Not All Factors Crowd Equally: Modeling, Measuring, and Trading on Alpha Decay

## Strategy Description
This notebook implements a quantitative trading strategy based on the paper "Not All Factors Crowd Equally: Modeling, Measuring, and Trading on Alpha Decay" by Chorok Lee. The strategy focuses on trading factors that exhibit hyperbolic decay in their alpha, particularly momentum and reversal factors. The strategy aims to capture the decay in factor alpha over time, adjusting positions based on the estimated decay.

## Paper Citation
Lee, Chorok. "Not All Factors Crowd Equally: Modeling, Measuring, and Trading on Alpha Decay." arXiv preprint arXiv:2512.11913 (2025).

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'FB', 'NFLX', 'NVDA']
HYPERBOLIC_DECAY_PARAMS = {'K': 1.0, 'lambda': 0.05}
SIGNAL_THRESHOLD = 0.5
POSITION_SIZING_FACTOR = 0.1

# Hypothesis
# The strategy hypothesizes that factors exhibiting hyperbolic decay in alpha can be traded profitably by adjusting positions based on the decay rate. Momentum and reversal factors are expected to show clear hyperbolic decay, allowing for timely entry and exit of positions.


## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download historical data
data = yf.download(UNIVERSE, start='2010-01-01', end='2023-12-31', group_by='ticker')

# Compute momentum factor
data['momentum'] = data['Adj Close'].pct_change(21)

# Compute reversal factor
data['reversal'] = -data['Adj Close'].pct_change(5)

# Cross-sectional normalization
data['momentum_z'] = data['momentum'].apply(lambda x: (x - x.mean()) / x.std())
data['reversal_z'] = data['reversal'].apply(lambda x: (x - x.mean()) / x.std())


## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Signal generation based on hyperbolic decay
def hyperbolic_decay(t, K, lambd):
    return K / (1 + lambd * t)

# Generate signals
data['signal'] = data['momentum_z'].apply(lambda x: 1 if x > SIGNAL_THRESHOLD else (-1 if x < -SIGNAL_THRESHOLD else 0))

# Position sizing
data['position'] = data['signal'] * POSITION_SIZING_FACTOR


## Phase 4 — Vectorized Backtest

In [ ]:
# Shift signals forward by 1 period to avoid look-ahead bias
data['position'] = data['position'].shift(1)

# Calculate daily returns
data['daily_return'] = data['Adj Close'].pct_change()

# Calculate portfolio returns
data['portfolio_return'] = data['daily_return'] * data['position']

# Cumulative returns
data['cumulative_return'] = (1 + data['portfolio_return']).cumprod()


## Phase 5 — Performance Metrics

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Calculate performance metrics
annual_return = data['portfolio_return'].mean() * 252
annual_volatility = data['portfolio_return'].std() * np.sqrt(252)
sharpe_ratio = annual_return / annual_volatility
sortino_ratio = annual_return / data['portfolio_return'][data['portfolio_return'] < 0].std() * np.sqrt(252)
max_drawdown = (data['cumulative_return'].cummax() - data['cumulative_return']).max()
calmar_ratio = annual_return / max_drawdown

# Print performance metrics
print(f'Annual Return: {annual_return:.2%}')
print(f'Annual Volatility: {annual_volatility:.2%}')
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2%}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')

# Plot equity curve
plt.plot(data['cumulative_return'])
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.show()


## Phase 6 — Monitoring Stub

In [ ]:
# Monitoring function
def monitor_portfolio(live_data):
    live_data['position'] = live_data['signal'].shift(1)
    live_data['daily_return'] = live_data['Adj Close'].pct_change()
    live_data['portfolio_return'] = live_data['daily_return'] * live_data['position']
    daily_pnl = live_data['portfolio_return'].iloc[-1]
    current_positions = live_data['position'].iloc[-1]
    print(f'Daily P&L: {daily_pnl:.2%}')
    print(f'Current Positions: {current_positions}')

# Example usage
monitor_portfolio(data)
